# LDDM — Large Drug Discovery Model

A pocket-conditioned generative model for structure-based drug design:
de novo design, fragment growing/linking, docking, partial docking,
programmable design and synthesizable design — all from one model.

- Code: https://github.com/LPDI-EPFL/lddm
- Paper: https://www.biorxiv.org/content/10.64898/2026.09.15.751537v1

This notebook runs inside the `lddm` kernel, which is the LDDM container.
The working directory is `/opt/lddm`. The folders `examples/`, `configs/`,
`checkpoints/`, `data/` and `output/` are bound in from project space, so
anything written there persists after the kernel stops. Everything else in
the container is read-only.

In [ ]:
!ls
# Output should match the file layout of https://github.com/LPDI-EPFL/lddm

In [ ]:
# Confirm the GPU is visible. Sampling works on CPU too (add --device cpu),
# but it is 5-10x slower.
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# The checkpoint is downloaded by build_lddm.sh, not baked into the image.
!ls -lh checkpoints/

## De novo design

Generate new molecules from scratch inside the KRAS pocket. The reference
ligand defines where the pocket is; it is not used as a starting structure.

https://github.com/LPDI-EPFL/lddm#de-novo-design

Runtime: 51 s measured on an L4.

In [ ]:
%run scripts/sample.py design \
    --protein examples/kras.pdb \
    --ref_ligand examples/kras_ref_ligand.sdf \
    --checkpoint checkpoints/lddm.ckpt \
    --output examples/de_novo_samples.sdf

## Fragment-based design (fragment growing)

Same as above, but `--ligand` supplies a fragment that is kept and grown from.

https://github.com/LPDI-EPFL/lddm#fragment-based-design

Runtime: not measured. Upstream quotes about 50 s on an H100; treat
that as indicative only, since the comparable de novo and docking cells
both ran faster than their upstream H100 figures on an L4.

In [ ]:
%run scripts/sample.py design \
    --protein examples/kras.pdb \
    --ref_ligand examples/kras_ref_ligand.sdf \
    --ligand examples/kras_frag.sdf \
    --checkpoint checkpoints/lddm.ckpt \
    --output examples/grown_samples.sdf

## Docking

In `dock` mode the molecule is fixed and only its pose is generated, so
`--molecule_size` does not apply.

https://github.com/LPDI-EPFL/lddm#docking

Runtime: 50 s measured on an L4.

In [ ]:
%run scripts/sample.py dock \
    --protein examples/kras.pdb \
    --ref_ligand examples/kras_ref_ligand.sdf \
    --ligand examples/kras_ref_ligand.sdf \
    --checkpoint checkpoints/lddm.ckpt \
    --output examples/docked_samples.sdf

## Partial docking

Only the atoms listed in `--atoms_to_dock` get new coordinates; every other
atom keeps its input position. This is the workflow for covalent ligands,
where the warhead stays put.

https://github.com/LPDI-EPFL/lddm#partial-docking

Runtime: not measured. Upstream quotes about 45 s on an H100; see the
note on the fragment growing cell above.

In [ ]:
%run scripts/sample.py dock \
    --protein examples/kras.pdb \
    --ref_ligand examples/kras_ref_ligand.sdf \
    --ligand examples/kras_ref_ligand.sdf \
    --atoms_to_dock 16 17 18 19 20 21 24 25 26 27 28 29 30 31 \
    --checkpoint checkpoints/lddm.ckpt \
    --output examples/partially_docked_samples.sdf

## Inspecting the output

Each run writes an SDF with `--n_samples` records (10 by default).

In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors

for path in ["examples/de_novo_samples.sdf",
             "examples/grown_samples.sdf",
             "examples/docked_samples.sdf",
             "examples/partially_docked_samples.sdf"]:
    mols = [m for m in Chem.SDMolSupplier(path) if m is not None]
    print(f"\n{path}  ({len(mols)} molecules)")
    for m in mols[:5]:
        print(f"  MW {Descriptors.MolWt(m):7.1f}  {Chem.MolToSmiles(m)}")

## Useful parameters

| Parameter | Mode | Default | Description |
|---|---|---|---|
| `--n_samples` | both | `10` | Number of molecules to generate |
| `--molecule_size` | design | histogram | Integer, `uniform_<low>_<high>`, or omit to sample from the pocket-conditioned size histogram |
| `--atoms_to_dock` | dock | whole molecule | Atom indices to re-place (partial docking) |
| `--n_steps` | both | `100` | Integration steps — higher is slower but usually better |
| `--sampler` | both | `ForwardEuler` | `ForwardEuler` or `HeunSampler` |
| `--sampling_noise` | both | `5.0` | Noise injected into coordinates during sampling |
| `--batch_size` | both | `--n_samples` | Lower this if you hit GPU OOM |
| `--seed` | both | `None` | Set for reproducible sampling |
| `--device` | both | `cuda:0` | Use `cpu` to run without a GPU |

A larger run, for example:

In [ ]:
%run scripts/sample.py design \
    --protein examples/kras.pdb \
    --ref_ligand examples/kras_ref_ligand.sdf \
    --checkpoint checkpoints/lddm.ckpt \
    --n_samples 100 \
    --batch_size 25 \
    --n_steps 200 \
    --sampler HeunSampler \
    --seed 42 \
    --output output/kras_100_samples.sdf

## Programmable design

Locally programmable design, driven by a YAML config rather than CLI flags.
Edit `configs/controlled_generation/programmable_design.yml` to change the
target, the constraints or the output path — it is bound in from project
space, so edits persist.

https://github.com/LPDI-EPFL/lddm#programmable-design

Runtime for the default KRAS example, measured on one NVIDIA L4:

| | First run | Later runs |
|---|---|---|
| Geometry reference compilation | 30 min | skipped (cached) |
| Sampling, 6 iterations | 7 min | 7 min |
| **Total** | **38 min** | **7 min** |

Peak memory was 4.9 GB, inside the 20 GB that `run_lddm.sl` requests.

Output is roughly 150 molecules in `output/kras_programmable_samples.sdf`,
plus a pocket PDB and a 96-column per-molecule CSV carrying every evaluator
score used for filtering.

> **The first run is slow.** The geometry evaluator compiles bond, angle,
> torsion and ring reference distributions from the 228 MB
> `data/validity3d/ligands.sdf`, which takes about half an hour of
> single-threaded CPU work before any sampling starts. It caches the result
> beside that file as `ligands_geometry_values.p` and
> `ligands_geometry_kernel_densities.p` (about 900 MB together). Because
> `data/` is bound in from project space the cache survives, and every later
> run — including the synthesizable design cell below — skips this step.

> **`seed: 42` does not make this bit-reproducible.** Two identical runs here
> produced 153 and 143 molecules. The seed fixes the sampler, but the
> iterative loop's fragment selection depends on evaluator scores computed on
> the GPU, and those are not deterministic. Expect the same character of
> result, not the same molecules.

In [ ]:
!cat configs/controlled_generation/programmable_design.yml

In [ ]:
%run scripts/generate_programmable_design.py configs/controlled_generation/programmable_design.yml

## Synthesizable design

Generates molecules constrained to a virtual chemical space, so the output is
synthesizable by construction. `build_lddm.sh` downloads the open SynSpace
chemical space (44,944 building blocks, 3 reactions) into `data/synspace/`;
this cell fails with a missing-file error if that download did not complete.

Enamine REAL needs a separate licence from Enamine and cannot be redistributed;
to use it, prepare it with `scripts/prepare_chemical_space.py` and point
`itergen_params` in the config at the result.

https://github.com/LPDI-EPFL/lddm#synthesizable-design

Runtime on one NVIDIA L4: **20 min 34 s**, peak memory 3.6 GB. That assumes
the geometry cache from the programmable design cell above already exists; on
a cold cache add about 30 minutes.

This config also switches on docking (`use_docking: true`), so each iteration
alternates between growing fragments and posing them, and it expands the
chemical space as it goes — the run below generated 40,940 new building blocks
in one iteration alone.

Output is about two dozen molecules in `output/kras_synthesizable_samples.sdf`
(the run measured here found 56 and kept 24 after the final filter). Unlike
programmable design, each molecule carries its synthesis route:

| Property | Meaning |
|---|---|
| `react_trace` | The reaction path, naming each reaction and the SynSpace building block id it consumed |
| `synthon_smiles` | The synthons the molecule decomposes into |
| `completed` | Whether a full route to purchasable building blocks was found |
| `is_docked` | Whether the final pose came from the docking stage |

A `react_trace` reads like this — a reductive amination between SynSpace
building block 35560 and the scaffold:

```
<<Cc1ccccc1C=O-synspace_35560>;<Cc1cncc(N2CCNCC2)c1-NA>:reductive amination:...>
```

In [ ]:
%run scripts/generate_programmable_design.py configs/controlled_generation/synthesizable_design.yml

In [ ]:
!ls -lh output/